## Clean raw multi-season match data
Parses the nested JSON-like `score`/`homeTeam`/`awayTeam` fields, flattens them into plain columns, filters to finished matches only, and saves the cleaned result.

In [1]:
import pandas as pd
import ast

matches_df = pd.read_csv("../data/raw/raw_matches_multi_season.csv")
teams_df = pd.read_csv("../data/raw/raw_teams.csv")
standings_df = pd.read_csv("../data/raw/raw_standings.csv")

## Step 2: Parse nested JSON-like fields
The raw API response stores `score`, `homeTeam`, and `awayTeam` as Python-dict-like strings — this parses them into real dicts, then flattens out the fields we actually need.

In [2]:
matches_df["score"] = matches_df["score"].apply(ast.literal_eval)
matches_df["homeTeam"] = matches_df["homeTeam"].apply(ast.literal_eval)
matches_df["awayTeam"] = matches_df["awayTeam"].apply(ast.literal_eval)

matches_df["home_team_id"] = matches_df["homeTeam"].apply(lambda x: x["id"])
matches_df["home_team_name"] = matches_df["homeTeam"].apply(lambda x: x["name"])
matches_df["away_team_id"] = matches_df["awayTeam"].apply(lambda x: x["id"])
matches_df["away_team_name"] = matches_df["awayTeam"].apply(lambda x: x["name"])

matches_df["home_goals"] = matches_df["score"].apply(lambda x: x["fullTime"]["home"])
matches_df["away_goals"] = matches_df["score"].apply(lambda x: x["fullTime"]["away"])
matches_df["winner"] = matches_df["score"].apply(lambda x: x["winner"])

matches_df["utcDate"] = pd.to_datetime(matches_df["utcDate"])

## Step 3: Filter to finished matches only

In [3]:
matches_df["status"].value_counts()  # see what's in there first
finished_matches = matches_df[matches_df["status"] == "FINISHED"].copy()

## Step 4: Select final columns and save

In [4]:
clean_matches_multi = finished_matches[[
    "id", "utcDate", "status", "matchday", "season",
    "home_team_id", "home_team_name", "away_team_id", "away_team_name",
    "home_goals", "away_goals", "winner"
]]

clean_matches_multi.to_csv("../data/processed/clean_matches_multi_season.csv", index=False)
print(clean_matches_multi.shape)

(1140, 12)
